# STOIC download on Google Colab

This notebook downloads the STOIC public training set (2000 CT scans) to:

`/content/drive/MyDrive/dataset/pretrain/STOIC/download`

Official command:

`aws s3 cp s3://stoic2021-training/ /path/to/destination/ --recursive --no-sign-request`

No AWS account is required when using `--no-sign-request`.

In [1]:
# 1) Mount Google Drive
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# 2) Prepare destination and ensure AWS CLI is available
import os
import shutil

DEST = '/content/drive/MyDrive/dataset/pretrain/STOIC/download'
os.makedirs(DEST, exist_ok=True)

# Colab usually already has aws installed; install only if missing.
if shutil.which('aws') is None:
    !pip -q install awscli

!aws --version
print('Destination:', DEST)

aws-cli/1.44.87 Python/3.12.13 Linux/6.6.113+ botocore/1.42.97
Destination: /content/drive/MyDrive/dataset/pretrain/STOIC/download


In [4]:
# 3) Download STOIC public training set (anonymous access)
!aws s3 cp s3://stoic2021-training/ "$DEST" --recursive --no-sign-request

download: s3://stoic2021-training/README.txt to drive/MyDrive/dataset/pretrain/STOIC/download/README.txt
download: s3://stoic2021-training/LICENCE.txt to drive/MyDrive/dataset/pretrain/STOIC/download/LICENCE.txt
download: s3://stoic2021-training/data/mha/10032.mha to drive/MyDrive/dataset/pretrain/STOIC/download/data/mha/10032.mha
download: s3://stoic2021-training/data/mha/10011.mha to drive/MyDrive/dataset/pretrain/STOIC/download/data/mha/10011.mha
download: s3://stoic2021-training/data/mha/1002.mha to drive/MyDrive/dataset/pretrain/STOIC/download/data/mha/1002.mha
download: s3://stoic2021-training/data/mha/10010.mha to drive/MyDrive/dataset/pretrain/STOIC/download/data/mha/10010.mha
download: s3://stoic2021-training/data/mha/1003.mha to drive/MyDrive/dataset/pretrain/STOIC/download/data/mha/1003.mha
download: s3://stoic2021-training/data/mha/10044.mha to drive/MyDrive/dataset/pretrain/STOIC/download/data/mha/10044.mha
download: s3://stoic2021-training/data/mha/10033.mha to drive/MyDr

In [5]:
# 4) Optional: verify files
!du -sh "$DEST"
!ls -lah "$DEST" | head -n 30

244G	/content/drive/MyDrive/dataset/pretrain/STOIC/download
total 28K
drwx------ 3 root root 4.0K Apr 29 04:36 data
-rw------- 1 root root  19K Nov 24  2021 LICENCE.txt
drwx------ 2 root root 4.0K Apr 29 05:41 metadata
-rw------- 1 root root  375 Nov 24  2021 README.txt


In [6]:
# 5) Check downloaded folder structure and .mha files
import os
from pathlib import Path

root = Path(DEST)
if not root.exists():
    raise FileNotFoundError(f'Directory does not exist: {root}')

print(f'Root: {root}')

# Print a compact tree (up to depth=2) to inspect the structure quickly.
max_depth = 2
for current, dirs, files in os.walk(root):
    rel = Path(current).relative_to(root)
    depth = len(rel.parts)
    if depth > max_depth:
        dirs[:] = []
        continue

    indent = '  ' * depth
    name = '.' if str(rel) == '.' else rel.name
    print(f'{indent}{name}/  (dirs={len(dirs)}, files={len(files)})')

# Count and preview .mha files recursively.
mha_files = sorted(root.rglob('*.mha'))
print(f'\nTotal .mha files: {len(mha_files)}')
if mha_files:
    print('Sample .mha files:')
    for p in mha_files[:20]:
        print(' -', p)
else:
    print('No .mha files found yet. If download just finished, verify the source contents and rerun.')

Root: /content/drive/MyDrive/dataset/pretrain/STOIC/download
./  (dirs=2, files=2)
  data/  (dirs=1, files=0)
    mha/  (dirs=0, files=2000)
  metadata/  (dirs=0, files=1)

Total .mha files: 2000
Sample .mha files:
 - /content/drive/MyDrive/dataset/pretrain/STOIC/download/data/mha/10010.mha
 - /content/drive/MyDrive/dataset/pretrain/STOIC/download/data/mha/10011.mha
 - /content/drive/MyDrive/dataset/pretrain/STOIC/download/data/mha/1002.mha
 - /content/drive/MyDrive/dataset/pretrain/STOIC/download/data/mha/1003.mha
 - /content/drive/MyDrive/dataset/pretrain/STOIC/download/data/mha/10032.mha
 - /content/drive/MyDrive/dataset/pretrain/STOIC/download/data/mha/10033.mha
 - /content/drive/MyDrive/dataset/pretrain/STOIC/download/data/mha/10036.mha
 - /content/drive/MyDrive/dataset/pretrain/STOIC/download/data/mha/10044.mha
 - /content/drive/MyDrive/dataset/pretrain/STOIC/download/data/mha/10058.mha
 - /content/drive/MyDrive/dataset/pretrain/STOIC/download/data/mha/10064.mha
 - /content/drive

In [7]:
# 6) Split all .mha into 4 folders: file1/file2/file3/file4
import shutil
from pathlib import Path

root = Path(DEST)
src_dir = root / 'data' / 'mha'

if not src_dir.exists():
    raise FileNotFoundError(f'Source directory not found: {src_dir}')

mha_files = sorted(src_dir.glob('*.mha'))
if not mha_files:
    raise RuntimeError(f'No .mha files found in: {src_dir}')

# Create target folders under download/
target_dirs = [root / f'file{i}' for i in range(1, 5)]
for d in target_dirs:
    d.mkdir(parents=True, exist_ok=True)

n = len(mha_files)
base = n // 4
extra = n % 4
sizes = [base + (1 if i < extra else 0) for i in range(4)]

print(f'Total .mha: {n}')
print('Planned split sizes:', sizes)

start = 0
for i, (dst_dir, size) in enumerate(zip(target_dirs, sizes), start=1):
    part = mha_files[start:start + size]
    for src in part:
        dst = dst_dir / src.name
        if dst.exists():
            # Keep operation idempotent for reruns: skip if already moved.
            continue
        shutil.move(str(src), str(dst))
    start += size
    count_now = len(list(dst_dir.glob('*.mha')))
    print(f'file{i}: {count_now} files')

remaining = len(list(src_dir.glob('*.mha')))
print(f'Remaining in source {src_dir}: {remaining}')

Total .mha: 2000
Planned split sizes: [500, 500, 500, 500]
file1: 500 files
file2: 500 files
file3: 500 files
file4: 500 files
Remaining in source /content/drive/MyDrive/dataset/pretrain/STOIC/download/data/mha: 0
